# Data quality

Checks on the stock-day panel before any signal is tested.
Nothing here uses future returns, so it cannot bias the pre-registered tests.

Run `scripts/build_panel.py` first.

In [1]:
import numpy as np
import pandas as pd

from shortvol import panel, prices

pd.set_option("display.width", 140)
DATA = prices.ROOT / "data"

df = pd.read_parquet(DATA / "processed" / "panel.parquet")
df["svr"] = df.short_volume / df.total_volume
df["off_exchange_share"] = df.total_volume / df.volume
df["dollar_volume"] = df.close * df.volume
df["size_tercile"] = df.groupby("date").dollar_volume.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["small", "mid", "large"])
)
df["year"] = df.date.dt.year

## 1. Coverage
How many S&P 500 member-days made it into the panel, and how often FINRA has no row.
A missing FINRA row means no off-exchange trading was reported, not zero shorting.

In [2]:
membership = prices.load_membership("2016-01-01", "2026-08-31")
days = pd.DatetimeIndex(sorted(df.date.unique()))
expected = len(panel.membership_days(membership, days))
per_day = df.groupby("date").size()
no_finra = df.short_volume.isna()

print(f"stock-days in panel: {len(df):,} of {expected:,} member-days ({len(df) / expected:.1%})")
print(f"stocks: {df.ticker.nunique()}, per day: min {per_day.min()}, median {per_day.median():.0f}")
print(f"no FINRA row: {no_finra.mean():.2%} of stock-days")
print("no FINRA row by size tercile:", no_finra.groupby(df.size_tercile, observed=True).mean().map("{:.2%}".format).to_dict())
print(f"price-source volume missing: {df.volume.isna().sum()} stock-days")

stock-days in panel: 1,317,970 of 1,352,000 member-days (97.5%)
stocks: 710, per day: min 468, median 493
no FINRA row: 0.48% of stock-days
no FINRA row by size tercile: {'small': '0.50%', 'mid': '0.59%', 'large': '0.33%'}
price-source volume missing: 90 stock-days


## 2. One observation
AAPL on 2023-03-07.
FINRA's consolidated number is the sum of the facility columns (Q = Nasdaq Carteret, N = NYSE, B = Nasdaq Chicago).
`volume` is consolidated volume from the price source, so off-exchange share = 22.4M / 56.2M, about 40%.

In [3]:
cols = ["date", "ticker", "short_volume", "short_exempt_volume", "total_volume",
        "TotalVolume_Q", "TotalVolume_N", "TotalVolume_B", "volume", "open", "close", "svr"]
df[(df.ticker == "AAPL") & (df.date == "2023-03-07")][cols].T

,8973
date,2023-03-07 00:00:00
ticker,AAPL
short_volume,9202982.0
short_exempt_volume,102147.0
total_volume,22405019.0
TotalVolume_Q,12378308.0
TotalVolume_N,9921962.0
TotalVolume_B,104749.0
volume,56182000.0
open,153.699997


## 3. Why a 10,000-share floor
SVR is a ratio, so on thin days a single dealer trade decides it.
This uses every FINRA symbol in 2023 (not just the S&P 500) to show where the noise sets in.
Below 10k shares, a large share of days read as almost entirely short; above it, noise declines slowly.
The floor is chosen from the signal's own noise, never from returns.

In [4]:
f = pd.read_parquet(
    DATA / "interim" / "finra_short_volume.parquet",
    columns=["date", "symbol", "short_volume", "total_volume"],
    filters=[("date", ">=", pd.Timestamp("2023-01-01")), ("date", "<=", pd.Timestamp("2023-12-31"))],
).sort_values(["symbol", "date"])
f["svr"] = f.short_volume / f.total_volume
f["day_jump"] = f.groupby("symbol").svr.diff().abs()
bins = [0, 1e3, 5e3, 1e4, 5e4, 1e5, 5e5, 1e6, 1e7, np.inf]
labels = ["<1k", "1-5k", "5-10k", "10-50k", "50-100k", "100-500k", "0.5-1M", "1-10M", ">10M"]
f["off_exchange_volume"] = pd.cut(f.total_volume, bins, labels=labels)
f.groupby("off_exchange_volume", observed=True).agg(
    stock_days=("svr", "size"),
    svr_std=("svr", "std"),
    median_day_jump=("day_jump", "median"),
    pct_days_95pct_short=("svr", lambda s: (s >= 0.95).mean() * 100),
).round(3)

,stock_days,svr_std,median_day_jump,pct_days_95pct_short
off_exchange_volume,,,,
<1k,252154,0.364,0.267,31.348
1-5k,304656,0.291,0.214,6.635
5-10k,194569,0.250,0.174,2.829
10-50k,568098,0.219,0.138,1.280
50-100k,268304,0.198,0.114,0.754
100-500k,540739,0.184,0.101,0.585
0.5-1M,140909,0.171,0.089,0.490
1-10M,157244,0.157,0.074,0.409
>10M,14847,0.113,0.050,0.168


In [5]:
below = df.total_volume < 10_000
print(f"S&P 500 stock-days below the floor: {below.mean():.3%}")

S&P 500 stock-days below the floor: 0.125%


## 4. The short volume ratio
Distribution, and how its level varies with size and over time.
Level shifts are removed later by ranking stocks against each other each day.

In [6]:
ok = df[~no_finra & ~below].copy()
print(ok.svr.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(3).to_string())
print("\nmean SVR by size tercile:", ok.groupby("size_tercile", observed=True).svr.mean().round(3).to_dict())
print("mean SVR by year:", ok.groupby("year").svr.mean().round(3).to_dict())

count    1310016.000
mean           0.468
std            0.147
min            0.009
1%             0.151
25%            0.362
50%            0.466
75%            0.571
99%            0.797
max            0.982



mean SVR by size tercile: {'small': 0.498, 'mid': 0.47, 'large': 0.435}
mean SVR by year: {2016: 0.459, 2017: 0.438, 2018: 0.423, 2019: 0.44, 2020: 0.445, 2021: 0.442, 2022: 0.484, 2023: 0.501, 2024: 0.492, 2025: 0.516, 2026: 0.516}


## 5. Where the variation is
If most variation were a fixed level per stock, the signal would mostly re-sort stocks by type.
Most of it is within a stock over time, which is what a daily signal needs.

In [7]:
stock_mean = ok.groupby("ticker").svr.transform("mean")
share_between = stock_mean.var() / ok.svr.var()
ok = ok.sort_values(["ticker", "date"])
autocorr = ok.svr.corr(ok.groupby("ticker").svr.shift(1))
print(f"between stocks: {share_between:.0%}, within a stock over time: {1 - share_between:.0%}")
print(f"day-to-day autocorrelation of SVR: {autocorr:.2f}")

between stocks: 13%, within a stock over time: 87%
day-to-day autocorrelation of SVR: 0.61


## 6. Off-exchange share
FINRA volume / consolidated volume.
This is the conditioning variable for Thesis B (wholesaler and retail flow).
Values above 1 are price-source volume errors and are excluded when it is used.

In [8]:
share = ok.off_exchange_share.where(ok.off_exchange_share <= 1)
print(share.describe(percentiles=[0.01, 0.5, 0.99]).round(3).to_string())
print(f"\nabove 1 (excluded): {(ok.off_exchange_share > 1).mean():.3%} of stock-days")
print("median by year:", share.groupby(ok.year).median().round(3).to_dict())
print("median by size tercile:", share.groupby(ok.size_tercile, observed=True).median().round(3).to_dict())

count    1309780.000
mean           0.282
std            0.085
min            0.012
1%             0.107
50%            0.277
99%            0.501
max            0.983

above 1 (excluded): 0.015% of stock-days
median by year: {2016: 0.259, 2017: 0.248, 2018: 0.232, 2019: 0.24, 2020: 0.257, 2021: 0.253, 2022: 0.267, 2023: 0.292, 2024: 0.315, 2025: 0.356, 2026: 0.359}


median by size tercile: {'small': 0.253, 'mid': 0.276, 'large': 0.301}


## 7. Facility mix
Share of FINRA volume reported to each facility, by year.
Nasdaq Chicago (B) starts in 2018-08, a small break in the series.

In [9]:
fac = ok.filter(like="TotalVolume_").fillna(0).groupby(ok.year).sum()
(fac.div(fac.sum(axis=1), axis=0) * 100).round(1)

,TotalVolume_B,TotalVolume_N,TotalVolume_Q
year,,,
2016,0.0,11.7,88.3
2017,0.0,8.6,91.4
2018,0.0,15.0,85.0
2019,0.2,19.7,80.1
2020,0.5,21.2,78.4
2021,0.4,16.7,82.9
2022,0.5,15.0,84.5
2023,0.4,15.3,84.3
2024,0.5,4.4,95.1


## 8. Price-source validation
The same stock's FINRA volume and consolidated volume move together day to day.
Two or more consecutive quarters with correlation under 0.5 mean the price source served a different company under that ticker.
IR is the clearest case: Gardner Denver's history sits under IR until the ticker actually moved in 2020 Q1.

In [10]:
agree = pd.read_csv(DATA / "processed" / "volume_agreement.csv")
print(f"median quarterly volume correlation: {agree['corr'].median():.2f}")
print("wrong-company quarters dropped:", agree.wrong_company.sum(), sorted(agree[agree.wrong_company].ticker.unique()))
print("source mix (stock-days):", df.source.value_counts().to_dict())
agree[agree.ticker == "IR"].head(14)

median quarterly volume correlation: 0.86
wrong-company quarters dropped: 6 ['CTXS', 'DD', 'FLIR']
source mix (stock-days): {'yfinance': 1225353, 'tiingo': 92617}


,ticker,quarter,days,corr,wrong_company


## 9. Extreme moves
Open-to-close moves above 50% are checked by hand.
All are real events: First Republic's rescue day, Globe Life's short report, Moderna in August 2026.

In [11]:
r = df.close / df.open - 1
df[r.abs() > 0.5][["date", "ticker", "source", "open", "close", "volume"]]

,date,ticker,source,open,close,volume
504252,2023-03-16,FRC,tiingo,20.220000,34.270000,196079956.0
529721,2024-04-11,GL,yfinance,98.480003,49.169998,36577500.0
828597,2026-08-19,MRNA,yfinance,116.019997,174.380005,199252300.0
